# 05 - Comparators (Phase B3) and Cross-Platform Generalization (Phase C)

Two things this notebook exists to do, both blocked on the project's local
Windows machine and both trivially unblocked on Colab's Linux + bigger GPU:

1. **Phase B3 - real comparators.** STAGATE and Garfield have never run in this
   project. STAGATE is blocked because `torch_sparse` has no prebuilt wheel for
   torch 2.11.0+cu128 on Windows and fails to build from source. That is an
   environment problem, not a research problem. Until they run, the results
   table compares against GraphST alone, which is thin for a benchmark claim.
2. **Phase C - generalization.** Every number in this project so far comes from
   one tissue on one platform (DLPFC, 10x Visium). That is the single biggest
   objection to the work, and the paired significance analysis makes it sharper:
   at n=11 slices the per-seed effect size vs GraphST is large (r=-0.545) with a
   bootstrap CI of [-0.072, +0.005], i.e. GraphST is probably modestly ahead and
   11 slices cannot settle it. **More datasets, not more DLPFC tuning, is what
   resolves an underpowered comparison.**

**Design rule: this notebook CLONES the repo rather than inlining model code.**
The previous Colab notebook (`04_colab_scaleup.ipynb`) pasted a copy of the
Phase 0 `EmbeddedMemoryLayer` inline; that copy is now eight development stages
stale and would silently benchmark a model nobody uses. Cloning guarantees
these runs exercise the *current* architecture and the *same* evaluation
protocol as every local result.


## 0. Environment


In [ ]:
!nvidia-smi


In [ ]:
# Repo + core deps. Torch/CUDA already present on Colab.
!git clone -q https://github.com/AnkitDash-code/Memory_RECOMB-27.git repo
%cd repo/recomb2027
!pip install -q scanpy squidpy anndata scikit-learn scikit-misc pot GraphST entmax

import sys
sys.path.insert(0, '.')
import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)


## 1. Phase B3 - install the blocked comparators

Both were checked for installability before committing time to them
(the plan's own instruction). STAGATE resolves from its author's repo;
`garfield` is on PyPI as v1.0.0 and is confirmed to be the spatial-omics
package (Weige Zhou, github.com/zhou-1314/Garfield), not a name collision.


In [ ]:
# STAGATE needs PyG. On Colab these have prebuilt Linux wheels.
!pip install -q torch-geometric
!pip install -q git+https://github.com/QIFEIDKN/STAGATE_pyG.git
!pip install -q garfield


### 1a. Garfield API discovery

Garfield's README documents parameters but gives no concrete quickstart, so
rather than guess at an API (and risk writing confident-looking code that
does the wrong thing), discover it here and write the wrapper afterwards
against what is actually there.


In [ ]:
import garfield, inspect
print('version:', getattr(garfield, '__version__', 'unknown'))
print('top-level:', [n for n in dir(garfield) if not n.startswith('_')])
for name in [n for n in dir(garfield) if not n.startswith('_')][:15]:
    obj = getattr(garfield, name)
    if inspect.isclass(obj) or inspect.isfunction(obj):
        try:
            print(f'\n{name}{inspect.signature(obj)}')
        except (ValueError, TypeError):
            print(f'\n{name} (signature unavailable)')


### 1b. STAGATE on DLPFC, scored with the shared protocol

`src/models/run_stagate.py` follows STAGATE's own DLPFC tutorial
(3000 seurat_v3 HVGs, normalize/log1p, `rad_cutoff=150`) -- a comparator
should be run the way its authors run it. The resulting embedding then goes
through `src/eval/clustering.py`, the identical mclust-equivalent +
spatial-refinement protocol every other method here is scored with.


In [ ]:
import numpy as np
from sklearn.metrics import adjusted_rand_score

from src.data.load_dlpfc import ALL_DLPFC_SAMPLES, load_dlpfc_slice
from src.eval.clustering import cluster_embedding, consensus_cluster
from src.models.run_stagate import run_stagate

SEEDS = [0, 1, 2, 3, 4]
TUNING_SLICE = '151673'

stagate_results = []
for sample in ALL_DLPFC_SAMPLES:
    raw = load_dlpfc_slice(sample)
    truth = raw.obs['ground_truth_layer']
    mask = truth.notna().to_numpy()
    n_layers = int(truth.nunique())

    labels_per_seed, aris = [], []
    for seed in SEEDS:
        out = run_stagate(raw.copy(), device=DEVICE, random_seed=seed)
        labels = cluster_embedding(out.obsm['STAGATE'], n_layers,
                                   coords=out.obsm['spatial'], refine=True)
        labels_per_seed.append(labels)
        aris.append(adjusted_rand_score(truth[mask], np.asarray(labels)[mask]))

    cons = consensus_cluster(labels_per_seed, n_layers)
    stagate_results.append({
        'sample': sample, 'per_seed': aris,
        'mean': float(np.mean(aris)), 'std': float(np.std(aris)),
        'consensus': float(adjusted_rand_score(truth[mask], np.asarray(cons)[mask])),
    })
    print(stagate_results[-1], flush=True)

held = [r for r in stagate_results if r['sample'] != TUNING_SLICE]
print('\nSTAGATE held-out per-seed :', np.mean([r['mean'] for r in held]).round(4))
print('STAGATE held-out consensus:', np.mean([r['consensus'] for r in held]).round(4))


In [ ]:
import json, pathlib
pathlib.Path('outputs/logs').mkdir(parents=True, exist_ok=True)
json.dump(stagate_results, open('outputs/logs/stagate_dlpfc_results.json', 'w'), indent=2)
from google.colab import files; files.download('outputs/logs/stagate_dlpfc_results.json')


## 2. Phase C - cross-platform generalization

Three platforms, chosen so results are directly comparable to the literature
this work is measured against:

| dataset | platform | why |
|---|---|---|
| Mouse olfactory bulb | Stereo-seq | used in GraphST's own paper -> a literature number on the same data |
| Human breast cancer | 10x Visium | standard secondary benchmark (SpaGCN/STAGATE/GraphST all report it); different tissue, same platform -> isolates tissue effects |
| Mouse hippocampus | Slide-seqV2 | bead-based, not spot-array: different resolution and sparsity profile -> the strongest generalization test, since the *platform* differs |

**Methodological guardrail, carried over from the DLPFC work:** do not retune
the architecture per dataset to chase numbers. Re-running an *already-validated*
selection procedure is fine; introducing new per-dataset architectural changes
is the same leakage as tuning on the test set, merely spread across datasets
instead of slices. The DLPFC-validated defaults (`memory_slots=16`, `n_hops=4`,
`lambda_usage=0.02`, expression-weighted adjacency) are used as-is.

**What to watch for.** A *consistent* relative result across platforms is a
strong generalization story. An *inconsistent* one is equally valuable: the
pattern of where the method wins versus loses becomes the Phase D hypothesis,
exactly as the per-subject DLPFC breakdown did.


In [ ]:
# Slide-seqV2 is available directly through squidpy; the other two are
# downloaded per their sources. Start with Slide-seqV2 since it needs no
# manual download and is the strongest generalization test of the three.
import squidpy as sq

slide = sq.datasets.slideseqv2()
print(slide)
print('sparsity:', 1 - slide.X.nnz / (slide.X.shape[0] * slide.X.shape[1]))
print('has ground-truth labels:', [c for c in slide.obs.columns if 'cluster' in c.lower() or 'type' in c.lower()])


### 2a. A caveat to settle before reporting any Slide-seqV2 ARI

Slide-seqV2 has **cell-type** annotations, not **spatial-domain** annotations.
ARI against cell type measures a different task than ARI against cortical
layers, and the two are not interchangeable -- a spatial-domain method is not
supposed to recover cell types. Check what the comparator papers actually
score on this dataset before reporting a number, and if no domain annotation
exists, report it as an unsupervised-metric comparison (silhouette / spatial
coherence) rather than manufacturing a supervised score against the wrong
label set.


In [ ]:
# Runner for our model + GraphST on an arbitrary dataset, reusing the exact
# training and clustering code paths as the DLPFC evaluation.
from src.data.preprocess import preprocess_hvg
from src.models.train_spatial_address import train_spatial_address_model
from src.models.run_graphst import run_graphst


def evaluate_dataset(raw, n_clusters, label_key=None, seeds=(0, 1, 2, 3, 4),
                     coord_type='generic'):
    """coord_type='generic' for non-Visium (bead-based) platforms, where the
    Visium hex-grid assumption in preprocess_hvg does not hold."""
    adata = preprocess_hvg(raw.copy(), coord_type=coord_type)
    truth = adata.obs[label_key] if label_key else None
    mask = truth.notna().to_numpy() if truth is not None else None

    out = {}
    for name in ['ours', 'graphst']:
        labels_per_seed, aris = [], []
        for seed in seeds:
            if name == 'ours':
                _, tr, _ = train_spatial_address_model(adata.copy(), seed=seed,
                                                       device=DEVICE, verbose=False)
                emb, coords = tr.obsm['X_spatial_address'], tr.obsm['spatial']
            else:
                g = run_graphst(raw.copy(), n_clusters=n_clusters, device=DEVICE,
                                random_seed=seed, cluster=False)
                emb, coords = g.obsm['emb'], g.obsm['spatial']
            labels = cluster_embedding(emb, n_clusters, coords=coords, refine=True)
            labels_per_seed.append(labels)
            if truth is not None:
                aris.append(adjusted_rand_score(truth[mask], np.asarray(labels)[mask]))
        cons = consensus_cluster(labels_per_seed, n_clusters)
        out[name] = {
            'per_seed': aris,
            'mean': float(np.mean(aris)) if aris else None,
            'std': float(np.std(aris)) if aris else None,
            'consensus': (float(adjusted_rand_score(truth[mask], np.asarray(cons)[mask]))
                          if truth is not None else None),
        }
    return out


## 3. Reporting

Bring results back into the repo and score them with the *same* statistical
machinery as DLPFC -- `src/eval/significance_test.py` now reports paired
Wilcoxon **plus** rank-biserial effect size **plus** bootstrap CIs, because at
small n a p-value alone cannot distinguish 'no effect' from 'underpowered'.
Report every platform whether it wins, ties, or loses; the pattern is the
finding, and a loss on one platform is information, not a failure to hide.
